# Baseline Tabular Models - Online Retail

Baseline models: XGBoost and LightGBM for 30-day horizon.
Data source: `data/transform/online_retail_daily_product_tabular.csv`.

In [ ]:
# Install dependencies (run once per environment)
!pip install xgboost lightgbm catboost tensorflow scikit-learn darts neuralforecast pytorch-forecasting

In [ ]:
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

In [ ]:
MODEL_CONFIG = {
    "xgboost":    True,
    "lightgbm":   True,
    "catboost":   False,
    "ridge":      True,
    "elasticnet": True,
    "lstm":       False,
}

ACTIVE = [n for n, e in MODEL_CONFIG.items() if e]
print(f"Models: {len(ACTIVE)}/{len(MODEL_CONFIG)} active - {ACTIVE}")


In [ ]:
data_path = "../data/transform/online_retail_daily_product_tabular.csv"
df = pd.read_csv(data_path)
df["date"] = pd.to_datetime(df["date"])
df.head()

In [ ]:
df.shape

## Train/Test Split (Time-based)
Split per product: train = all dates before last 30 days, test = last 30 days.

In [ ]:
horizon_days = 30
df = df.sort_values(["stock_code", "date"]).reset_index(drop=True)

global_max_date = df["date"].max()
cutoff = global_max_date - pd.Timedelta(days=horizon_days)

train_mask = df["date"] <= cutoff
test_mask = df["date"] > cutoff

df_train = df.loc[train_mask].copy()
df_test = df.loc[test_mask].copy()

print("Global cutoff:", cutoff)
print("Train shape:", df_train.shape, "Test shape:", df_test.shape)

In [ ]:
df.columns

## Feature Preparation
- Compute avg_price from revenue/demand
- Create lag features: demand_lag_{1,7,14,28}, avg_price_lag_1, revenue_lag_1, num_invoices_lag_1
- Rolling stats (shifted by 1): roll_mean_{7,14,28}, roll_std_{7,28}
- Diff features: diff_1, diff_7
- Static product features (train-only): product_popularity, product_revenue_share, product_lifecycle_age
- Calendar features: day_of_week, week_of_year, month, quarter, etc.
- **All features computed BEFORE train/test split or shifted to prevent leakage**


In [ ]:
target_col = "demand_qty"

# ============================================================
# Step 1: Ensure sorted by product + date
# ============================================================
df = df.sort_values(["stock_code", "date"]).reset_index(drop=True)

# ============================================================
# Step 2: Compute avg_price (before any lag features)
# ============================================================
df["avg_price"] = df["revenue"] / df["demand_qty"].replace(0, np.nan)
df["avg_price"] = df.groupby("stock_code")["avg_price"].transform(lambda x: x.ffill().bfill().fillna(0))

# ============================================================
# Step 3: Lag features (all use .shift() -- no same-day leakage)
# ============================================================

# Demand lag
for lag in [1, 2, 7, 14, 28]:
    df[f"demand_lag_{lag}"] = df.groupby("stock_code")["demand_qty"].shift(lag)

# Price/revenue/invoice lag-1
df["avg_price_lag_1"] = df.groupby("stock_code")["avg_price"].shift(1)
mean_prices = df.groupby("stock_code")["avg_price"].transform("mean")
df["avg_price_lag_1"] = df["avg_price_lag_1"].fillna(mean_prices).fillna(0)

df["revenue_lag_1"] = df.groupby("stock_code")["revenue"].shift(1).fillna(0)
df["num_invoices_lag_1"] = df.groupby("stock_code")["num_invoices"].shift(1).fillna(0)

# ============================================================
# Step 4: Rolling features (shifted by 1 -- no leakage)
# ============================================================

for window in [7, 14, 28]:
    df[f"roll_mean_{window}"] = (
        df.groupby("stock_code")["demand_qty"]
        .shift(1)
        .rolling(window=window, min_periods=1)
        .mean()
    )

for window in [7, 28]:
    df[f"roll_std_{window}"] = (
        df.groupby("stock_code")["demand_qty"]
        .shift(1)
        .rolling(window=window, min_periods=1)
        .std()
    )

# Diff features (lag-based, no target leakage)
df["diff_1"] = df["demand_lag_1"] - df["demand_lag_2"]
df["diff_7"] = df["demand_lag_7"] - df["demand_lag_14"]

# ============================================================
# Step 5: Static product features (computed from TRAIN ONLY)
# ============================================================

# Use global cutoff (already defined in Cell 7)
df_train_only = df[df["date"] <= cutoff].copy()

# product_popularity: total demand per product (train only)
prod_demand = df_train_only.groupby("stock_code")["demand_qty"].sum()
df["product_popularity"] = df["stock_code"].map(prod_demand).fillna(0).astype(int)

# product_revenue_share: share of total revenue (train only)
prod_rev = df_train_only.groupby("stock_code")["revenue"].sum()
total_rev = prod_rev.sum()
if total_rev > 0:
    df["product_revenue_share"] = df["stock_code"].map(prod_rev) / total_rev
else:
    df["product_revenue_share"] = 0.0
df["product_revenue_share"] = df["product_revenue_share"].fillna(0)

# product_lifecycle_age: days since first sale (train only)
first_sale = df_train_only.groupby("stock_code")["date"].min()
df["first_sale_date"] = df["stock_code"].map(first_sale)
df["product_lifecycle_age"] = (df["date"] - df["first_sale_date"]).dt.days
df["product_lifecycle_age"] = df["product_lifecycle_age"].fillna(0).clip(lower=0)
df = df.drop(columns=["first_sale_date"])

# ============================================================
# Step 6: Fill remaining NaN in lag/rolling columns
# ============================================================

lag_roll_cols = [c for c in df.columns if any(
    c.startswith(p) for p in ["demand_lag", "avg_price_lag", "revenue_lag",
                              "num_invoices_lag", "roll_mean", "roll_std", "diff"]
)]
df[lag_roll_cols] = df[lag_roll_cols].fillna(0)

# ============================================================
# Step 7: Split train/test (re-do with enriched features)
# ============================================================

train_mask = df["date"] <= cutoff
test_mask = df["date"] > cutoff

df_train = df.loc[train_mask].copy()
df_test = df.loc[test_mask].copy()

# Save target BEFORE dropping columns from X
y_train = df_train["demand_qty"].copy()
y_test = df_test["demand_qty"].copy()


df_train["stock_code_raw"] = df_train["stock_code"]
df_test["stock_code_raw"] = df_test["stock_code"]

for col in ["stock_code", "product_revenue_total"]:
    if col in df_train.columns:
        df_train = df_train.drop(columns=[col])
    if col in df_test.columns:
        df_test = df_test.drop(columns=[col])

# ============================================================
# Step 8: Define expanded leakage-free feature set
# ============================================================

FEATURE_COLS = [
    "day_of_week", "week_of_year", "month", "quarter", "day_of_month",
    "is_weekend", "is_month_start", "is_month_end",
    "demand_lag_1", "demand_lag_2", "demand_lag_7", "demand_lag_14", "demand_lag_28",
    "avg_price_lag_1", "revenue_lag_1", "num_invoices_lag_1",
    "roll_mean_7", "roll_mean_14", "roll_mean_28",
    "roll_std_7", "roll_std_28",
    "diff_1", "diff_7",
    "product_popularity", "product_revenue_share", "product_lifecycle_age",
]

feature_cols = [col for col in FEATURE_COLS if col in df_train.columns]

print(f"Features ({len(feature_cols)}) - leakage-free for batch prediction:")
for col in feature_cols:
    print(f"  [OK] {col}")

X_train = df_train[feature_cols].copy()
X_test = df_test[feature_cols].copy()

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

print(f"\nX_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Zero-demand: train={(y_train==0).mean()*100:.1f}%, test={(y_test==0).mean()*100:.1f}%")

X_train.shape, X_test.shape


In [ ]:
X_train.head(5)

In [ ]:
X_train.columns

## Baseline Model: XGBoost


In [ ]:
if MODEL_CONFIG["xgboost"]:
    with tqdm(total=100, desc="[XGBoost]", unit="%") as pbar:
        pbar.set_postfix_str("Initializing...")
        pbar.update(10)
        pbar.set_postfix_str("Fitting 300 trees...")
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8, objective="reg:squarederror", random_state=42)
        xgb_model.fit(X_train, y_train)
        xgb_preds = xgb_model.predict(X_test)
        pbar.update(90)
        pbar.set_postfix_str("Done")
else:
    xgb_model = None
    xgb_preds = np.zeros(len(X_test))
    with tqdm(total=100, desc="[XGBoost]", unit="%") as pbar:
        pbar.set_postfix_str("SKIPPED - set to True to enable")
        pbar.update(100)


## Baseline Model: LightGBM


In [ ]:
if MODEL_CONFIG["lightgbm"]:
    with tqdm(total=100, desc="[LightGBM]", unit="%") as pbar:
        pbar.set_postfix_str("Initializing...")
        pbar.update(10)
        pbar.set_postfix_str("Fitting 300 trees...")
        lgbm_model = LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31,
            max_depth=-1, subsample=0.8, colsample_bytree=0.8, random_state=42)
        lgbm_model.fit(X_train, y_train)
        lgbm_preds = lgbm_model.predict(X_test)
        pbar.update(90)
        pbar.set_postfix_str("Done")
else:
    lgbm_model = None
    lgbm_preds = np.zeros(len(X_test))
    with tqdm(total=100, desc="[LightGBM]", unit="%") as pbar:
        pbar.set_postfix_str("SKIPPED - set to True to enable")
        pbar.update(100)


## Additional Model: CatBoost


In [ ]:
if MODEL_CONFIG["catboost"]:
    with tqdm(total=100, desc="[CatBoost]", unit="%") as pbar:
        pbar.set_postfix_str("Initializing...")
        pbar.update(10)
        pbar.set_postfix_str("Fitting...")
        cat_model = CatBoostRegressor(verbose=False)
        cat_model.fit(X_train, y_train)
        cat_preds = cat_model.predict(X_test)
        pbar.update(90)
        pbar.set_postfix_str("Done")
else:
    cat_model = None
    cat_preds = np.zeros(len(X_test))
    with tqdm(total=100, desc="[CatBoost]", unit="%") as pbar:
        pbar.set_postfix_str("SKIPPED - set to True to enable")
        pbar.update(100)


## Additional Model: Ridge Regression


In [ ]:
if MODEL_CONFIG["ridge"]:
    with tqdm(total=100, desc="[Ridge]", unit="%") as pbar:
        pbar.set_postfix_str("Fitting...")
        pbar.update(30)
        ridge_model = Ridge()
        ridge_model.fit(X_train, y_train)
        ridge_preds = ridge_model.predict(X_test)
        pbar.update(70)
        pbar.set_postfix_str("Done")
else:
    ridge_model = None
    ridge_preds = np.zeros(len(X_test))
    with tqdm(total=100, desc="[Ridge]", unit="%") as pbar:
        pbar.set_postfix_str("SKIPPED - set to True to enable")
        pbar.update(100)


## Additional Model: ElasticNet


In [ ]:
if MODEL_CONFIG["elasticnet"]:
    with tqdm(total=100, desc="[ElasticNet]", unit="%") as pbar:
        pbar.set_postfix_str("Fitting...")
        pbar.update(30)
        elastic_model = ElasticNet()
        elastic_model.fit(X_train, y_train)
        elastic_preds = elastic_model.predict(X_test)
        pbar.update(70)
        pbar.set_postfix_str("Done")
else:
    elastic_model = None
    elastic_preds = np.zeros(len(X_test))
    with tqdm(total=100, desc="[ElasticNet]", unit="%") as pbar:
        pbar.set_postfix_str("SKIPPED - set to True to enable")
        pbar.update(100)


## LSTM Baseline (TensorFlow/Keras)


In [ ]:
if MODEL_CONFIG["lstm"]:
    time_steps = 7
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    def create_seq(Xs, ys, df_s, steps):
        for code, grp in df_s.groupby("stock_code_raw"):
            gl = len(grp)
            if gl <= steps: continue
            idx = grp.index.tolist()
            Xg = Xs[idx]; yg = ys[idx]
            for i in range(len(Xg) - steps):
                yield Xg[i:i+steps], yg[i+steps]

    df_ts = df_train.sort_values(["stock_code_raw", "date"]).reset_index(drop=True)
    df_tes = df_test.sort_values(["stock_code_raw", "date"]).reset_index(drop=True)

    Xtr_seq = []; ytr_seq = []
    for Xb, yb in create_seq(X_train_scaled, y_train.values, df_ts, time_steps):
        Xtr_seq.append(Xb); ytr_seq.append(yb)
    Xtr_seq = np.array(Xtr_seq); ytr_seq = np.array(ytr_seq)

    Xte_seq = []; yte_seq = []
    for Xb, yb in create_seq(X_test_scaled, y_test.values, df_tes, time_steps):
        Xte_seq.append(Xb); yte_seq.append(yb)
    Xte_seq = np.array(Xte_seq); yte_seq = np.array(yte_seq)

    print(f"Seq: train={Xtr_seq.shape}, test={Xte_seq.shape}")

    lstm_model = Sequential([LSTM(64, input_shape=(time_steps, Xtr_seq.shape[2]), return_sequences=True),
        Dropout(0.2), LSTM(32), Dropout(0.2), Dense(1)])
    lstm_model.compile(optimizer="adam", loss="mae")

    lstm_preds_all = np.zeros(len(df_tes))
    with tqdm(total=20, desc="[LSTM]", unit="epoch") as pbar:
        for ep in range(20):
            lstm_model.fit(Xtr_seq, ytr_seq, epochs=1, batch_size=64, verbose=0)
            pbar.update(1)
            loss = lstm_model.history.history["loss"][-1]
            pbar.set_postfix_str(f"loss={loss:.4f}")

        pbar.set_postfix_str("Predicting...")
        pred_flat = lstm_model.predict(Xte_seq, verbose=0).flatten()

        si = 0
        for code, grp in df_tes.groupby("stock_code_raw"):
            gl = len(grp); idx = grp.index.tolist()
            if gl <= time_steps:
                fb = df_train[df_train["stock_code_raw"]==code]["demand_qty"].mean()
                lstm_preds_all[idx] = fb if not np.isnan(fb) else 0; continue
            np_ = gl - time_steps
            lstm_preds_all[idx[:time_steps]] = grp["demand_lag_1"].fillna(0).values[:time_steps]
            lstm_preds_all[idx[time_steps:]] = pred_flat[si:si+np_]; si += np_
        pbar.set_postfix_str("Done")
else:
    lstm_model = None
    lstm_preds_all = np.zeros(len(df_test))
    with tqdm(total=100, desc="[LSTM]", unit="%") as pbar:
        pbar.set_postfix_str("SKIPPED - set to True to enable")
        pbar.update(100)


## Evaluation Metrics

### Metric Definitions

| Metric | Direction | What it Measures |
|--------|-----------|-----------------|
| MAE | ↓ smaller=better | Rata-rata selisih absolut forecast vs actual |
| RMSE | ↓ smaller=better | Seperti MAE tapi menghukum error besar lebih keras |
| SMAPE (%) | ↓ smaller=better | Error persentase simetris (0-200%), aman untuk data nol |
| F1 Zero | ↑ larger=better | Akurasi mendeteksi demand nol vs non-nol |
| MAE NonZero | ↓ smaller=better | MAE hanya pada hari dengan demand > 0 |
| CLS ($) | ↓ smaller=better | Biaya lost sales akibat under-forecast |
| IHC ($) | ↓ smaller=better | Biaya holding inventory akibat over-forecast |
| OOS Rate | ↓ smaller=better | Seberapa sering forecast < actual |
| OFR | ↑ larger=better | Persentase demand yang terpenuhi oleh forecast |
| FVA (%) | ↑ larger=better | % improvement dibanding naive baseline |
| Peak Capture (%) | ↑ larger=better | Seberapa baik model menangkap hari puncak |
| Peak WAPE | ↓ smaller=better | Weighted error pada hari puncak |
| Peak Bias (%) | → near 0=better | Kecenderungan over/under-forecast di hari puncak |
| Peak CLS ($) | ↓ smaller=better | Lost sales cost khusus hari puncak |

**Arah:** ↓ = makin kecil makin baik · ↑ = makin besar makin baik · → = makin dekat 0 makin baik


In [ ]:
gross_margin = 0.20
holding_cost_annual = 0.20
holding_cost_daily = holding_cost_annual / 365
cogs_ratio = 1 - gross_margin


In [ ]:
# Safe CLS/IHC functions
def calc_cls_safe(y_true, y_pred, avg_price_arr, margin):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    avg_price = np.asarray(avg_price_arr, dtype=float)
    avg_price = np.nan_to_num(avg_price, nan=0.0, posinf=0.0, neginf=0.0)
    return np.sum(np.maximum(y_true - y_pred, 0) * avg_price * margin)

def calc_ihc_safe(y_true, y_pred, avg_price_arr, cogs, holding_daily):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    avg_price = np.asarray(avg_price_arr, dtype=float)
    avg_price = np.nan_to_num(avg_price, nan=0.0, posinf=0.0, neginf=0.0)
    return np.sum(np.maximum(y_pred - y_true, 0) * avg_price * cogs * holding_daily)

def smape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom != 0
    if mask.sum() == 0: return 0
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100

def f1_zero(y_true, y_pred):
    yt = (y_true > 0).astype(int)
    yp = (y_pred > 0).astype(int)
    tp = np.sum((yp == 1) & (yt == 1))
    fp = np.sum((yp == 1) & (yt == 0))
    fn = np.sum((yp == 0) & (yt == 1))
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

# Build evaluation DataFrame
df_eval = df_test.copy()
df_eval["xgb_pred"] = xgb_preds
df_eval["lgbm_pred"] = lgbm_preds
df_eval["cat_pred"] = cat_preds
df_eval["ridge_pred"] = ridge_preds
df_eval["elastic_pred"] = elastic_preds
df_eval["lstm_pred"] = lstm_preds_all

# Attach true target back (was dropped from X)
df_eval["demand_qty"] = y_test.values


df_eval = df_eval.sort_values(["stock_code_raw", "date"]).reset_index(drop=True)
df_eval["naive_pred"] = df_eval.groupby("stock_code_raw")["demand_qty"].shift(1)
df_eval["naive_pred"] = df_eval["naive_pred"].fillna(0)
# Cold-start lookup: last train demand per product
last_demand = {}
for code in df_train["stock_code_raw"].unique():
    mask = df_train["stock_code_raw"] == code
    last_demand[code] = int(y_train.loc[df_train[mask].index[-1]])

for code, group in df_eval.groupby("stock_code_raw"):
    first_idx = group.index[0]
    if df_eval.loc[first_idx, "naive_pred"] == 0:
        df_eval.loc[first_idx, "naive_pred"] = last_demand.get(code, 0)


naive_mae = mean_absolute_error(df_eval["demand_qty"].values, df_eval["naive_pred"].values)

# ============================================================
# Single Consolidated Metrics Table
# ============================================================

y_test_vals = df_eval["demand_qty"].values
avg_price_arr = df_eval["avg_price_lag_1"].replace([np.inf,-np.inf],np.nan).fillna(0).values
avg_price_arr = np.nan_to_num(avg_price_arr, nan=0.0, posinf=0.0, neginf=0.0)

model_preds = {
    "xgboost": xgb_preds,
    "lightgbm": lgbm_preds,
    "catboost": cat_preds,
    "ridge": ridge_preds,
    "elasticnet": elastic_preds,
    "lstm": lstm_preds_all,
}

print("=== Consolidated Model Evaluation ===")
rows = []
with tqdm(total=len(MODEL_CONFIG), desc="Evaluating") as pbar:
    for mname in MODEL_CONFIG.keys():
        yp = model_preds[mname]
        enabled = MODEL_CONFIG[mname]
        if not enabled:
            rows.append({"Model": mname, "status": "N/A"})
            pbar.update(1)
            pbar.set_postfix_str(f"{mname}: N/A (disabled)")
            continue

        mae = mean_absolute_error(y_test_vals, yp)
        rmse = np.sqrt(mean_squared_error(y_test_vals, yp))
        s = smape(y_test_vals, yp)
        f1 = f1_zero(y_test_vals, yp)

        nz = y_test_vals > 0
        mae_nz = mean_absolute_error(y_test_vals[nz], yp[nz]) if nz.sum() > 0 else 0

        cls = calc_cls_safe(y_test_vals, yp, avg_price_arr, gross_margin)
        ihc = calc_ihc_safe(y_test_vals, yp, avg_price_arr, cogs_ratio, holding_cost_daily)
        oos = np.mean(yp < y_test_vals)
        demand_sum = y_test_vals.sum()
        ofr = np.minimum(y_test_vals, yp).sum() / demand_sum if demand_sum > 0 else 0
        fva = (1 - mae / naive_mae) * 100 if naive_mae > 0 else 0

# Peak metrics
        p95 = np.percentile(y_test_vals, 95)
        pk = y_test_vals > p95
        if pk.sum() > 0:
            pt = y_test_vals[pk].sum()
            pp = yp[pk].sum()
            pk_cap = (pp / pt) * 100 if pt > 0 else 0
            pk_wape = np.sum(np.abs(y_test_vals[pk] - yp[pk])) / pt if pt > 0 else 0
            pk_bias = (np.sum(yp[pk] - y_test_vals[pk]) / pt) * 100 if pt > 0 else 0
            pk_cls = calc_cls_safe(y_test_vals[pk], yp[pk], avg_price_arr[pk], gross_margin)
        else:
            pk_cap = pk_wape = pk_bias = pk_cls = 0

        rows.append({
            "Model": mname,
            "MAE": round(mae, 2), "RMSE": round(rmse, 2), "SMAPE_pct": round(s, 2),
            "F1_Zero": round(f1, 3), "MAE_NonZero": round(mae_nz, 2),
            "CLS": round(cls, 2), "IHC": round(ihc, 2),
            "OOS_Rate": round(oos, 3), "OFR": round(ofr, 3), "FVA_pct": round(fva, 1),
            "Peak_Capture_pct": round(pk_cap, 1), "Peak_WAPE": round(pk_wape, 3),
            "Peak_Bias_pct": round(pk_bias, 1), "Peak_CLS": round(pk_cls, 2),
        })

        pbar.update(1)
        pbar.set_postfix_str(f"{mname}: MAE={mae:.2f}")

results = pd.DataFrame(rows)
# Display table
print("\n" + "="*120)
print("CONSOLIDATED MODEL EVALUATION")
print("="*120)

display_cols = ["Model", "MAE", "RMSE", "SMAPE_pct", "F1_Zero", "MAE_NonZero",
                "CLS", "IHC", "OOS_Rate", "OFR", "FVA_pct",
                "Peak_Capture_pct", "Peak_WAPE", "Peak_Bias_pct", "Peak_CLS"]


cols_present = [c for c in display_cols if c in results.columns]

# Green highlight with dark text for contrast
def highlight_green_best(col, lower_better=True):
    styles = [""] * len(col)
    vals = pd.to_numeric(col, errors="coerce")
    if vals.isna().all():
        return styles
    if lower_better:
        best_idx = vals.idxmin()
    else:
        best_idx = vals.idxmax()
    styles[best_idx] = "background-color: #d4edda; color: #155724; font-weight: bold"
    return styles

styled = results[cols_present].style
for col in ["MAE", "RMSE", "SMAPE_pct", "CLS", "OOS_Rate", "Peak_WAPE"]:
    if col in results.columns:
        styled = styled.apply(highlight_green_best, subset=[col], lower_better=True)
for col in ["F1_Zero", "OFR", "FVA_pct", "Peak_Capture_pct"]:
    if col in results.columns:
        styled = styled.apply(highlight_green_best, subset=[col], lower_better=False)
display(styled)

print("\nLegend: green = best value | MAE/RMSE/SMAPE/CLS/OOS/Peak_WAPE = lower better | F1/OFR/FVA/Peak_Capture = higher better")
print("="*120)

# BUG FIX VERIFICATION
print("\nBUG FIX VERIFICATION")
print(f"Features: {len(feature_cols)} leakage-free: {feature_cols}")
print("CLS: Using .values + np.nan_to_num (no index mismatch)")


## Sample Plots (Actual vs Forecast)

In [ ]:
import matplotlib.pyplot as plt

top_10_products = (
    df_eval.groupby("stock_code_raw")["demand_qty"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

plot_df = df_eval[["stock_code_raw", "date", "demand_qty", "xgb_pred", "lgbm_pred", "cat_pred", "ridge_pred", "elastic_pred", "lstm_pred"]].copy()

for product in top_10_products:
    sub = plot_df[plot_df["stock_code_raw"] == product].sort_values("date")
    plt.figure(figsize=(10, 4))
    plt.plot(sub["date"], sub["demand_qty"], label="actual")
    plt.plot(sub["date"], sub["xgb_pred"], label="xgboost")
    plt.plot(sub["date"], sub["lgbm_pred"], label="lightgbm")
    plt.plot(sub["date"], sub["cat_pred"], label="catboost")
    plt.plot(sub["date"], sub["ridge_pred"], label="ridge")
    plt.plot(sub["date"], sub["elastic_pred"], label="elasticnet")
    plt.plot(sub["date"], sub["lstm_pred"], label="lstm")
    plt.title(f"Stock Code {product}: Actual vs Forecast")
    plt.xlabel("date")
    plt.ylabel("demand_qty")
    plt.legend()
    plt.tight_layout()
    plt.show()